In [1]:
#라이브러리 임포트
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
#데이터 불러오기
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data',train=True,transform=transform,download=True)
test_dataset = datasets.MNIST(root='./data',train=False,transform=transform,download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=16, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.66MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.20MB/s]


In [11]:
#모델, 손실함수 및 옵티마이저 정의

class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1,32,3,1,1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,1,1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Flatten(),
            nn.Linear(3136,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        return self.layers(x)
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MyCNN().to(device)
print(torch.cuda.is_available())

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(),lr=0.001)


True


In [12]:
#학습루프

num_epochs = 10

for i in range(num_epochs):
    for images, labels in train_loader:
        #GPU활용
        images, labels = images.to(device), labels.to(device)
        #기울기 초기화
        optimizer.zero_grad()
        #순전파
        y_pred = model(images)
        #손실함수계산
        loss = loss_fn(y_pred, labels)
        #역전파
        loss.backward()
        #가중치업데이트
        optimizer.step()
    # epoch마다 출력
    print(f'Epoch {i+1}/{num_epochs}, Loss: {loss.item():.4f}')

Epoch 1/10, Loss: 0.0582
Epoch 2/10, Loss: 0.0004
Epoch 3/10, Loss: 0.0004
Epoch 4/10, Loss: 0.0188
Epoch 5/10, Loss: 0.0001
Epoch 6/10, Loss: 0.0002
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0314
Epoch 10/10, Loss: 0.0001


In [14]:
#테스트

total = 0
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        #GPU활용
        images, labels = images.to(device), labels.to(device)
        test = model(images) #테스트 완료
        pred = torch.argmax(test, dim=1) #테스트 값중 가장 큰 값 추출
        total += labels.size(0) #데이터 전체 개수
        correct += (labels == pred).sum().item()

print(f'정확도: {100 * correct / total:.2f}%')

정확도: 99.08%
